# Introduction

This timeseries script provides the functions used to create timeserie profiles for industrial heat and passenger transport for when sufficient data is lacking. The script is specifically written for the global data set used to model 11 global regions in GENeSYS-MOD. Please note that necessary adjustments has to be made in order to apply the functions for other regional modeling set ups. Each function is preseded with a description of the assumptions behind the timeserie profile. 

To use this script, you need to have the following packages installed:
- numpy
- os
- pandas

In [35]:
import numpy as np
import os
import pandas as pd

## Passenger mobility

Here follows the function for creating the timeserie profile for passenger mobility. The functionality is struktured in foure pieces:
1. Base diurnal profiles - these are the base profiles for a 24h sequence. This includes one standard profile for weekdays (assuming more travel during peak commuting hours, with a slight decrease midday and a greater decrease during night), weekends (assuming less travel with more during day and afternoon), profiles for regions assumed to be more heavily car dependent or in contrast more dependent on public transport as well as a profile for regions with less travel during the day due to high temperature climate.
2. Function - this is the function which creates the annual profile bassed on the diurnal profiles and seasonality
3. User input - allows the user to specify year, outputfile and regions.
4. Run function - runs the function, creating the desired timeseries.

### Base Diurnal Profiles (24-Hour Local Time Kernels) - Used for the region definitions

In [29]:
wk_standard = np.array([ # Standard profile for weekend transport with morning and evening rushes
    0.008, 0.005, 0.004, 0.004, 0.008, 0.020, 0.045, 0.082, 0.088, 0.055,
    0.048, 0.046, 0.048, 0.050, 0.055, 0.068, 0.089, 0.092, 0.075, 0.052,
    0.040, 0.032, 0.023, 0.014
])
    
wk_car_heavy = np.array([ # Profile for regions with a car heavy transport profile, with clearer peaks and drops
    0.005, 0.003, 0.002, 0.002, 0.006, 0.022, 0.055, 0.095, 0.098, 0.048,
    0.042, 0.040, 0.042, 0.045, 0.052, 0.075, 0.098, 0.102, 0.065, 0.042,
    0.032, 0.025, 0.018, 0.010
])

wk_transit_heavy = np.array([ # Profile for regions with high modal share of public transit, with a flatter profile
    0.003, 0.002, 0.001, 0.001, 0.005, 0.025, 0.060, 0.085, 0.080, 0.060,
    0.055, 0.055, 0.055, 0.055, 0.060, 0.070, 0.085, 0.085, 0.075, 0.065,
    0.050, 0.035, 0.020, 0.008
])

wk_hot_climate = np.array([ # Profile for hot climate regions with assumed less mobility during warmest hours and extended evening activity
    0.012, 0.008, 0.005, 0.003, 0.005, 0.015, 0.035, 0.065, 0.070, 0.050,
    0.040, 0.035, 0.030, 0.032, 0.040, 0.055, 0.070, 0.075, 0.075, 0.075,
    0.085, 0.080, 0.060, 0.030
])

we_standard = np.array([ # Standard weekend profile with peak during early afternoon
    0.015, 0.010, 0.007, 0.005, 0.005, 0.008, 0.015, 0.025, 0.045, 0.065,
    0.078, 0.085, 0.088, 0.085, 0.080, 0.075, 0.070, 0.065, 0.055, 0.045,
    0.038, 0.032, 0.026, 0.018
])

### Function

In [36]:
def generate_genesysmod_mobility_profiles(year, output_file, regions):
    dates = pd.date_range(start=f"{year}-01-01 00:00", end=f"{year}-12-31 23:00", freq="h")
    
    hours = dates.hour
    dayofweek = dates.dayofweek  # 0=Mon, 1=Tue, ..., 4=Fri, 5=Sat, 6=Sun
    dayofyear = dates.dayofyear
    months = dates.month

    # Seasonality maps
    north_seasonality = {1: 0.92, 2: 0.94, 3: 0.98, 4: 1.00, 5: 1.02, 6: 1.06, 7: 1.10, 8: 1.12, 9: 1.04, 10: 1.00, 11: 0.95, 12: 1.02}
    south_seasonality = {1: 1.12, 2: 1.10, 3: 1.04, 4: 1.00, 5: 0.95, 6: 0.92, 7: 0.92, 8: 0.94, 9: 0.98, 10: 1.00, 11: 1.02, 12: 1.08}

    # -------------------------------------------------------------------------
    # Calculations
    # -------------------------------------------------------------------------
    df_out = pd.DataFrame()
    df_out["HOUR"] = np.arange(1, 8761)  

    for reg, meta in regions.items():
        # A. Seasonality - sets seasonal factors to increase/decrease 24h profile depending on month of year
        s_map = south_seasonality if meta["hemisphere"] == "south" else north_seasonality
        seasonal_factors = np.array([s_map[m] for m in months])
        
        # B. Special cultural surges - adjusts seasonal factors to take certain holidays into account for additional seasonal changes
        if meta.get("special_holiday") == "chunyun":
            chunyun_mask = (dayofyear >= 32) & (dayofyear <= 71)
            seasonal_factors[chunyun_mask] *= 1.35

        # C. Day type mask - creates regional profile taking weekdays into account and utilising the chosen standard week profile, with weekends assumed to have less travel
        is_weekend = np.isin(dayofweek, meta["weekend_days"])
        wk_prof = meta["diurnal_wk"] / meta["diurnal_wk"].sum()
        we_prof = meta["diurnal_we"] / meta["diurnal_we"].sum()
        
        raw_local = np.where(
            ~is_weekend,
            wk_prof[hours],
            we_prof[hours] * 0.85
        ) * seasonal_factors

        # D. Timezone shifting to UTC - shifts the profile from local time to UTC in order to get one cohesive dataset for UTC time
        shift = meta["utc_shift"]
        if shift > 0:
            raw_utc = np.concatenate([raw_local[shift:], raw_local[:shift]])
        elif shift < 0:
            s = abs(shift)
            raw_utc = np.concatenate([raw_local[-s:], raw_local[:-s]])
        else:
            raw_utc = raw_local

        # E. Standardize to Mean = 1.0 - adjusts profile to be normalised to the annual transport
        multiplier_profile = raw_utc / np.mean(raw_utc)
        df_out[reg] = np.round(multiplier_profile, 9)



    # Write CSV 
    print(f"Generated GENeSYS-MOD compatible file: {output_file}")
    os.makedirs(f"output/{year}", exist_ok=True)
    df_out.to_csv(f"output/{year}/{output_file}", index=False)
    return df_out

### User input

This is for the user to specify year of timeseries, where to store output and what regions to create timeseries for. 

For the global scenario with 11 regions it is assumed that Africa, Europe, FSU, Central & South America are using the standard week day diurnal profile. "wk_car_heavy" is used for North America and Oceania due to greater dependency on personal vehicles for passenger transport. Meanwhile "wk_transit_heavy" is used for China and Japan + Republic of Korea due to a well built out high-speed trains and urban rails. "wk_hot_climate" is applied to regions assumed to be overall heavily affected by high ambient temperatures during the day, decreasing travel at those hours and shifting it further into the evenings, used for Asia-Rest, India and Middle East.

Due to the great inpact of travel within China during the spring festival, it is assumed to increase the trvael within the country with 35% during the time of the festival.

In [31]:
# Decide for what year to run the function as well as name of output file
year=2018
output_file="global_regionalized_mobility_profiles_2018.csv"

# Provide regions - specify:
# - "diurnal_wk", use from the list of profiles 
# - "diurnal_we", uses the profile we_standard
# - "weekend_days", for days using weekend profile - where 0 = Monday and 6 = Sunday
# - "hemisphere", signifying seasonality - being either "north" or "south"
# - "utc_shift", in which timezone the region is placed in order to shift time series to one adherent UTC series - for simplicity assume a representative timezone for regions spanning multiple timezones
# - "special_holiday", for inclusion of Chinese spring festival celebration due to heavy increase in travel - currently no other holidays supported, assumed to have less profound effect on system or already included in Gregorian calander seasonality
regions = {
        "Africa":                {"diurnal_wk": wk_standard,
                                  "diurnal_we": we_standard,
                                  "weekend_days": [5, 6], 
                                  "hemisphere": "south", 
                                  "utc_shift":  2},
        "Asia-Rest":             {"diurnal_wk": wk_hot_climate,   
                                  "diurnal_we": we_standard,
                                  "weekend_days": [5, 6], 
                                  "hemisphere": "north", 
                                  "utc_shift":  7},
        "China":                 {"diurnal_wk": wk_transit_heavy, 
                                  "diurnal_we": we_standard,
                                  "weekend_days": [5, 6], 
                                  "hemisphere": "north", 
                                  "utc_shift":  8, 
                                  "special_holiday": "chunyun"},
        "Europe":                {"diurnal_wk": wk_standard,      
                                  "diurnal_we": we_standard,
                                  "weekend_days": [5, 6], 
                                  "hemisphere": "north", 
                                  "utc_shift":  1},
        "FSU":                   {"diurnal_wk": wk_standard,      
                                  "diurnal_we": we_standard,
                                  "weekend_days": [5, 6], 
                                  "hemisphere": "north", 
                                  "utc_shift":  3},
        "India":                 {"diurnal_wk": wk_hot_climate,   
                                  "diurnal_we": we_standard,
                                  "weekend_days": [6],    
                                  "hemisphere": "north", 
                                  "utc_shift":  5}, 
        "Japan + Republic of Korea":           {"diurnal_wk": wk_transit_heavy, 
                                                "diurnal_we": we_standard,
                                                "weekend_days": [5, 6], 
                                                "hemisphere": "north", 
                                                "utc_shift":  9},
        "Central & South America": {"diurnal_wk": wk_standard,      
                                    "diurnal_we": we_standard,
                                    "weekend_days": [5, 6], 
                                    "hemisphere": "south", 
                                    "utc_shift": -3},
        "Middle East":           {"diurnal_wk": wk_hot_climate,   
                                  "diurnal_we": we_standard,
                                  "weekend_days": [4, 5], 
                                  "hemisphere": "north", 
                                  "utc_shift":  3},
        "North America":         {"diurnal_wk": wk_car_heavy,     
                                  "diurnal_we": we_standard,
                                  "weekend_days": [5, 6],
                                  "hemisphere": "north", 
                                  "utc_shift": -5},
        "Oceania":               {"diurnal_wk": wk_car_heavy,      
                                  "diurnal_we": we_standard,
                                  "weekend_days": [5, 6], 
                                  "hemisphere": "south", 
                                  "utc_shift": 10},
    }

### Function run

In [37]:
df_mobility = generate_genesysmod_mobility_profiles(year, output_file, regions)

Generated GENeSYS-MOD compatible file: global_regionalized_mobility_profiles_2018.csv


## High industrial heat

Here follows the function for creating the timeserie profile for high industrial heat. The functionality is struktured in three pieces:
1. Function - this is the function which creates the annual profile bassed on  diurnal profiles and seasonality created within function
2. User input - allows the user to specify year, outputfile and regions.
3. Run function - runs the function, creating the desired timeseries.

### Function

In [38]:
def generate_genesysmod_heat_high_profiles(year,output_file, regions):
   
    dates = pd.date_range(start=f"{year}-01-01 00:00", end=f"{year}-12-31 23:00", freq="h")
    hours = dates.hour
    months = dates.month

    # Base Diurnal Multipliers by Season (24-Hour Local Time Kernels)
    # Winter profile: Continuous baseload with day/evening staff shifts
    diurnal_winter = np.array([
        1.82, 1.60, 1.38, 1.15, 0.93, 0.71, 0.49, 0.27, 0.40, 1.05,
        0.66, 0.80, 0.79, 0.78, 0.77, 0.76, 0.75, 0.90, 1.05, 1.21,
        1.36, 1.52, 1.67, 1.82
    ])

    # Summer profile: Lower overall baseload with daytime operational shift
    diurnal_summer = np.array([
        1.01, 0.94, 0.87, 0.80, 0.74, 0.67, 0.60, 0.53, 0.80, 0.77,
        0.65, 1.40, 1.33, 1.25, 1.17, 1.10, 1.02, 1.05, 1.07, 1.10,
        1.12, 1.15, 1.17, 1.20
    ])

    # Calculations

    df_out = pd.DataFrame()
    df_out["HOUR"] = np.arange(1, 8761)

    for reg, meta in regions.items():
        is_north = meta["hemisphere"] == "north"
        
        # Season selection: Northern (Winter = Q1/Q4) vs Southern (Winter = May-Sep)
        if is_north:
            is_winter_month = np.isin(months, [1, 2, 3, 10, 11, 12])
        else:
            is_winter_month = np.isin(months, [5, 6, 7, 8, 9])
            
        raw_local = np.where(is_winter_month, diurnal_winter[hours], diurnal_summer[hours])
        
        # Circular array shift to align local timeline to UTC
        shift = meta["utc_shift"]
        if shift > 0:
            raw_utc = np.concatenate([raw_local[shift:], raw_local[:shift]])
        elif shift < 0:
            s = abs(shift)
            raw_utc = np.concatenate([raw_local[-s:], raw_local[:-s]])
        else:
            raw_utc = raw_local
            
        # Standardize to exact annual unit mean (Mean = 1.000000)
        multiplier_profile = raw_utc / np.mean(raw_utc)
        df_out[reg] = np.round(multiplier_profile, 9)

    # Write CSV
    os.makedirs(f"output/{year}", exist_ok=True)
    df_out.to_csv(f"output/{year}/{output_file}", index=False)

    print(f"Generated GENeSYS-MOD compatible file: {output_file}")
    print(f"Dimensions: {df_out.shape[0]} rows x {df_out.shape[1]} columns")
    return df_out

### User input

This is for the user to specify year of timeseries, where to store output and what regions to create timeseries for. 

The pattern of industrial heat is assumed to be relatively constant over the year, not taking certain days or holidays into account since industry depending on high temperature heat normally must maintain a constant production in order to not decreasue the lifetime of plants and factories due to thermal inertia and shocks. 

In [45]:
# Decide for what year to run the function as well as name of output file
year = 2018
output_file = "global_regionalized_heat_high_profiles_2018.csv"

# Provide regions - specify:
# - "utc_shift", in which timezone the region is placed in order to shift time series to one adherent UTC series - for simplicity assume a representative timezone for regions spanning multiple timezones
# - "hemisphere", signifying seasonality - being either "north" or "south"

regions = {
        "Africa":                {"utc_shift":  2, "hemisphere": "south"},
        "Asia-Rest":             {"utc_shift":  7, "hemisphere": "north"},
        "China":                 {"utc_shift":  8, "hemisphere": "north"},
        "Europe":                {"utc_shift":  1, "hemisphere": "north"},
        "FSU":                   {"utc_shift":  3, "hemisphere": "north"},
        "India":                 {"utc_shift":  5, "hemisphere": "north"},
        "Japan + Republic of Korea":           {"utc_shift":  9, "hemisphere": "north"},
        "Central & South America": {"utc_shift": -3, "hemisphere": "south"},
        "Middle East":           {"utc_shift":  3, "hemisphere": "north"},
        "North America":         {"utc_shift": -5, "hemisphere": "north"},
        "Oceania":               {"utc_shift": 10, "hemisphere": "south"},
    }

### Run function

In [46]:
df_heat_high = generate_genesysmod_heat_high_profiles(year, output_file, regions)

Generated GENeSYS-MOD compatible file: global_regionalized_heat_high_profiles_2018.csv
Dimensions: 8760 rows x 12 columns
